<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.08 Making Models Smaller

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 1. Introduction

The models that make the headlines live in data centers. A frontier model holds hundreds of billions of parameters and needs a rack of accelerators to answer a single question. The Raspberry Pi 5 on your desk has 8 GB of memory, no accelerator, and a power budget of a few watts. Everything you have already run on it (`gemma3:1b-it-qat`, `qwen3:1.7b`, `deepseek-r1:1.5b`) exists only because someone deliberately made a larger model smaller.

This notebook is about that deliberate act. It sits between two notebooks you have already worked through: [`05.07_Introduction_to_Local_LLMs.ipynb`](./05.07_Introduction_to_Local_LLMs.ipynb) asked what a device is able to do, and [`05.07_Introduction_to_Local_LLMs.ipynb`](./05.07_Introduction_to_Local_LLMs.ipynb) listed which models we can put on it. The question in between is **what exactly is given up when a model is squeezed onto a device and how much of it can be won back by making a different compression choice?**

Four techniques do nearly all of the work in practice.

| Technique | What it changes | What it costs |
|:----------|:----------------|:--------------|
| **Quantization** | stores every weight with fewer bits | a small, controllable loss of precision |
| **Pruning** | deletes the weights that contribute least | usually needs retraining, and rarely shrinks the file as much as the name suggests |
| **Distillation** | trains a small model to imitate a large one | a full training run |
| **Selective parameter activation** | keeps every weight but uses only a few per token | saves computation, not memory |

Rather than describing these techniques, we will apply them to a real model, break it on purpose, and measure what breaks first. Everything in Sections 2 to 7 runs on your own laptop in a few minutes. Section 8 then checks whether the same effects show up on the Raspberry Pi.

<div class="alert alert-block alert-success">

Make sure that you have installed all necessary packages. If you are missing some, you can either use `pip install` or the provided cell in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb) (run the cell in section 1.2: "Install the Course Libraries").
</div>

#### 1.1 Why size is the binding constraint

Size matters for three separate reasons and they are worth keeping apart because they fail in different ways:

**Memory:** The weights must fit into RAM next to the operating system and everything else running on the device. If they do not fit, either the model simply refuses to load or the system becomes unusable. This is a hard wall, not a gradual slowdown.

**Bandwidth:** To produce a *single* token, the machine has to read every active weight out of memory once. Generation is therefore limited by how fast memory can be read, not by how fast the processor can multiply:

$$
\text{tokens per second} \;\approx\; \frac{\text{memory bandwidth}}{\text{bytes of active weights}}
$$

Halving the number of bytes roughly doubles the generation speed. Compression is thus not only a storage trick but rather the main speed lever available on a memory-bound device.

**Energy:** Moving a byte from memory to the processor costs considerably more energy than the arithmetic performed on it. Fewer bytes therefore means a longer battery life and on a passively cooled device it also means less throttling.

> For a manager the consequence is compact: on mobile hardware, model size is simultaneously the storage cost, the latency, and the battery cost. A decision to use a smaller model is not only a decision about quality.

### 2. Where the Memory Goes

The starting point is a multiplication that anyone can do:

$$
\text{weight memory} \;=\; \text{number of parameters} \;\times\; \text{bytes per parameter}
$$

The second factor is the one we control. A model is trained in 32-bit floating point (`FP32`), which spends 4 bytes on every single weight. Everything below that is a compression decision:

| Format | Bytes per weight | Typical role |
|:-------|:----------------:|:-------------|
| `FP32` | 4 | the format models are trained in |
| `FP16` / `BF16` | 2 | the standard format for serving in a data center |
| `INT8` | 1 | comfortable on-device format, almost free in quality |
| `INT4` | 0.5 | the format most models on your Pi actually use |

When Ollama labels a model `Q4_K_M`, the `4` is exactly this number: four bits per weight. We will find out in Section 3 what the `K_M` part is doing.

Let us compute the table for the models you have on the Pi. The Raspberry Pi 5 has 8 GB, of which roughly 2 GB are taken by the operating system and the runtime, so about 6 GB is realistically available.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BYTES_PER_WEIGHT = {"FP32": 4.0, "FP16": 2.0, "INT8": 1.0, "INT4": 0.5}

# Parameter counts of models from 05.07_Introduction_to_Local_LLMs.ipynb, plus two data-center models.
MODELS = {
    "tinyStories:110m":   0.11e9,
    "gemma3:270m":        0.27e9,
    "gemma3:1b-it-qat":   1.00e9,
    "deepseek-r1:1.5b":   1.50e9,
    "qwen3:1.7b":         1.70e9,
    "gemma3:4b-it-qat":   4.30e9,
    "llama3.1:8b":        8.00e9,
    "llama3.1:70b":      70.00e9,
    "a frontier model":  600.00e9,
}

PI_MEMORY_BUDGET_GB = 6.0   # 8 GB board, roughly 2 GB reserved for the operating system


def weight_memory_gb(n_parameters, precision):
    return n_parameters * BYTES_PER_WEIGHT[precision] / 1e9


footprint = pd.DataFrame(
    {precision: [weight_memory_gb(n, precision) for n in MODELS.values()]
     for precision in BYTES_PER_WEIGHT},
    index=list(MODELS),
)
footprint.index.name = "model"

print("Memory needed for the weights, in GB\n")
print(footprint.round(2))

print(f"\n\nDoes it fit into {PI_MEMORY_BUDGET_GB:.0f} GB of usable Raspberry Pi memory?\n")
print(footprint.map(lambda gb: "yes" if gb <= PI_MEMORY_BUDGET_GB else "-"))

# The same table as a picture: one row per model, one dot per precision.

PRECISION_COLORS = {"FP32": "#c7e9c0", "FP16": "#74c476", "INT8": "#238b45", "INT4": "#00441b"}

fig, ax = plt.subplots(figsize=(10, 4.8))
for precision, color in PRECISION_COLORS.items():
    ax.scatter(footprint[precision], range(len(footprint)), s=80, color=color,
               edgecolor="white", linewidth=0.8, label=precision, zorder=3)

ax.axvline(PI_MEMORY_BUDGET_GB, color="#d62728", linewidth=1.2, linestyle="--", zorder=2)
ax.text(PI_MEMORY_BUDGET_GB * 1.15, 0.2, "usable Pi memory (6 GB)", color="#d62728", fontsize=9)

ax.set_xscale("log")
ax.set_yticks(range(len(footprint)), footprint.index)
ax.set_xlabel("memory needed for the weights (GB, log scale)")
ax.invert_yaxis()
ax.grid(axis="x", color="#dddddd", linewidth=0.6, zorder=0)
ax.legend(title="precision", loc="lower right")
ax.set_title("Every dot left of the line fits on the Raspberry Pi", loc="left")
fig.tight_layout()
plt.show()

The chart is the table drawn out: each dot is one model at one precision and the red line is the wall. Read it by row and by column. Along a row, moving from `FP32` to `INT4` divides the memory by eight, and that is what turns `llama3.1:8b` from an impossible 32 GB into a 4 GB model that fits. Along a column, note where the "yes" turns into a "-": at `INT4` the Pi can hold an 8-billion-parameter model, while at `FP32` it cannot even hold a 1.7-billion-parameter one. The compression decision moves you two full model generations up the quality ladder on unchanged hardware.

The frontier-model row is there to keep a sense of proportion. No amount of quantization brings it onto this device. For that class of model, compression is not the answer, a network connection to a cloud solution is.

#### 2.1 The context window is not free either

The weights are not the only thing in memory. While a model generates, it also keeps a **KV cache**: for every token it has already seen, it stores the key and value vectors of every layer so it does not have to recompute them. That cache grows linearly with the length of the conversation:

$$
\text{cache bytes} \;=\; 2 \;\times\; \text{layers} \;\times\; \text{KV heads} \;\times\; \text{head dimension} \;\times\; \text{tokens} \;\times\; \text{bytes per value}
$$

The leading 2 counts the keys and the values. This is the hidden cost behind the `num_ctx` parameter you experimented with in [`05.11_Understanding_Inference_Control.ipynb`](./05.11_Understanding_Inference_Control.ipynb): raising the context window does not just slow the model down, it consumes memory that the weights would otherwise have used.

In [ ]:
def kv_cache_mb(n_layers, n_kv_heads, head_dim, n_tokens, bytes_per_value=2):
    """Memory the model spends on remembering the conversation so far."""
    return 2 * n_layers * n_kv_heads * head_dim * n_tokens * bytes_per_value / 1e6


# Shape of a typical 1B-class model. `ollama show <MODEL>` reports these numbers for yours.
N_LAYERS, N_KV_HEADS, HEAD_DIM = 26, 1, 256

for n_tokens in (512, 2048, 4096, 8192):
    cache = kv_cache_mb(N_LAYERS, N_KV_HEADS, HEAD_DIM, n_tokens)
    print(f"num_ctx = {n_tokens:5d} tokens -> {cache:7.1f} MB of KV cache")

<div class="alert alert-block alert-info">


### 3. Quantization: Fewer Bits per Weight

Quantization is the technique that actually ships. Every small model on your Raspberry Pi is quantized and if you only remember one method from this notebook, this should be it.

#### 3.1 What a trained weight matrix actually looks like

The idea rests on an observation about trained networks: the weights are not spread evenly over their range. They form a narrow bell around zero, with a very small number of far-away outliers. If the values were spread uniformly, we would need every bit of precision. Since they cluster, a coarse grid of a few values placed sensibly captures nearly all of the information.

We will work with `all-MiniLM-L6-v2`, the same sentence-embedding model you used in [`05.02_Understanding_Word_Embedding.ipynb`](./05.02_Understanding_Word_Embedding.ipynb). It is small enough to compress repeatedly within a notebook and it does a job we can score without a human in the loop: putting related words near each other.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

torch.set_num_threads(4)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
encoder = model[0].auto_model

# Keep an untouched copy so that every experiment can start from the original model.
ORIGINAL_WEIGHTS = {name: p.detach().clone() for name, p in encoder.named_parameters()}
N_PARAMETERS = sum(p.numel() for p in encoder.parameters())


def is_matrix(name, parameter):
    """The large 2-D weight matrices: what every real compression pipeline touches."""
    return parameter.dim() == 2 and parameter.numel() >= 4096


def is_linear(name, parameter):
    """Matrices inside the transformer layers, excluding the vocabulary lookup tables."""
    return is_matrix(name, parameter) and "embeddings" not in name


def restore_original():
    """Undo every compression experiment."""
    with torch.no_grad():
        for name, parameter in encoder.named_parameters():
            parameter.copy_(ORIGINAL_WEIGHTS[name])


in_matrices = sum(p.numel() for n, p in encoder.named_parameters() if is_matrix(n, p))
in_layers = sum(p.numel() for n, p in encoder.named_parameters() if is_linear(n, p))

print(f"parameters in total        : {N_PARAMETERS/1e6:6.1f} M")
print(f"held in large 2-D matrices : {in_matrices/1e6:6.1f} M  ({in_matrices/N_PARAMETERS:.1%} of the model)")
print(f"   inside transformer layers : {in_layers/1e6:6.1f} M")
print(f"   inside lookup tables      : {(in_matrices-in_layers)/1e6:6.1f} M")
print(f"stored as FP32             : {N_PARAMETERS*4/1e6:6.1f} MB")

Two things in that output are worth pausing on. Almost the entire model sits in a handful of two-dimensional matrices, which is why compression only ever targets those. The biases and normalization parameters are numerically delicate and too small to be worth touching. In a model this size, roughly half the parameters are the vocabulary lookup table rather than the transformer itself. For small models, the vocabulary is a major cost item.

#### 3.2 Rounding onto a grid

Quantization with `b` bits means we are allowed $2^b$ distinct values. The simplest and most common scheme is **symmetric absmax quantization**: find the largest magnitude in the matrix, spread the available levels evenly from `-max` to `+max` and round every weight to the nearest one. We store the small integers plus one floating-point `scale` and multiply back when the weight is needed.

With 4 bits we get the integers −7 to +7. The whole method is three lines of arithmetic:

In [ ]:
def quantize_dequantize(W, bits, group_size=None):
    """Round a weight matrix onto a coarse grid, then decode it back to floating point.

    group_size=None  -> one shared scale for the entire matrix
    group_size=128   -> one scale for every 128 neighbouring weights
    """
    levels = 2 ** (bits - 1) - 1                 # 4 bits -> the integers -7 ... +7
    groups = W.reshape(-1, group_size) if group_size else W.reshape(1, -1)
    scale = groups.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / levels
    codes = torch.clamp(torch.round(groups / scale), -levels, levels)
    return (codes * scale).reshape(W.shape)


EXAMPLE = "encoder.layer.0.intermediate.dense.weight"
W = ORIGINAL_WEIGHTS[EXAMPLE]

print(f"{EXAMPLE}   shape {tuple(W.shape)}")
print(f"  standard deviation : {W.std():.4f}")
print(f"  largest magnitude  : {W.abs().max():.4f}  "
      f"({W.abs().max()/W.std():.0f} standard deviations away from zero)\n")

for bits in (8, 4):
    for group_size in (None, 128, 32):
        recovered = quantize_dequantize(W, bits, group_size)
        error = (recovered - W).abs().mean() / W.abs().mean()
        collapsed = (recovered == 0).float().mean()
        label = "per-tensor" if group_size is None else f"group of {group_size}"
        print(f"  {bits}-bit, {label:14s} -> average error {error:6.1%}, "
              f"{collapsed:5.1%} of weights rounded to zero")

The 8-bit rows look harmless. The 4-bit per-tensor row does not: the average weight is reproduced with an error of the same order as the weight itself and the overwhelming majority of the matrix has been flattened to exactly zero. Something in this matrix is defeating the method.

#### 3.3 Why a single scale fails and what groups fix

The culprit is visible in the numbers printed above: the largest weight in the matrix sits more than twenty standard deviations from zero. Absmax quantization is forced to stretch its grid all the way out to that outlier, so the spacing between neighbouring grid points becomes larger than the entire bell in which almost every weight lives. Every one of those weights then rounds to the nearest grid point, which is zero.

The fix is to stop sharing one scale across half a million weights. **Group-wise quantization** gives each block of neighbouring weights (typically 32, 64 or 128 of them) its own scale. An outlier then only damages its own group and every other group gets a grid matched to its own range. This is what the `K_M` in Ollama's `Q4_K_M` refers to: a block-wise 4-bit format, not a single scale for the whole tensor.

The price is honest and small: one extra scale value per group. At a group size of 128 that adds 16 bits per 128 weights, about 0.125 bits per weight. We will see that show up in the file sizes.

In [ ]:
import matplotlib.pyplot as plt

GROUP_SIZE = 128
BITS = 4
levels = 2 ** (BITS - 1) - 1

all_weights = W.flatten().numpy()
step_tensor = float(W.abs().max()) / levels
grid_tensor = np.arange(-levels, levels + 1) * step_tensor

one_group = W.reshape(-1, GROUP_SIZE)[0].numpy()
step_group = float(np.abs(one_group).max()) / levels
grid_group = np.arange(-levels, levels + 1) * step_group

rounded_to_zero = float(np.mean(np.abs(all_weights) < step_tensor / 2))
outliers = int(np.sum(np.abs(all_weights) > 0.5))

print(f"{all_weights.size:,} weights in this matrix, of which {outliers} are larger than 0.5")
print(f"the largest of them is {np.abs(all_weights).max():.2f} and it alone sets the grid spacing")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)

axes[0].hist(all_weights, bins=300, color="#4c78a8")
for level in grid_tensor:
    axes[0].axvline(level, color="#d62728", linestyle="--", linewidth=0.9)
axes[0].set_yscale("log")
axes[0].set_title(
    f"One scale for the whole matrix: step = {step_tensor:.3f}\n"
    f"{rounded_to_zero:.0%} of all weights fall inside the first step and become zero",
    loc="left", fontsize=10)
axes[0].set_xlabel("weight value")
axes[0].set_ylabel("number of weights (log scale)")

axes[1].hist(one_group, bins=40, color="#4c78a8")
for level in grid_group:
    axes[1].axvline(level, color="#1b9e77", linestyle="--", linewidth=0.9)
axes[1].set_title(
    f"One scale per group of {GROUP_SIZE}: step = {step_group:.3f}\n"
    f"the same 15 levels now have to cover only this group",
    loc="left", fontsize=10)
axes[1].set_xlabel("weight value")
axes[1].set_ylabel("number of weights")

plt.show()

The left panel is the whole argument in one picture. The vertical scale is logarithmic, so the handful of weights out near ±1.5 are visible at all. There are only a few dozen of them among more than half a million. Yet those few dozen dictate where all fifteen red grid lines are placed, and the result is that the first grid interval swallows the entire bell.

The right panel shows one group of 128 weights with its own scale. The same fifteen levels now sit inside the range where the data actually is.

> This is a general lesson that reaches well beyond quantization: a compression scheme calibrated on the extreme values of a distribution will serve the typical case badly. Outliers deserve their own treatment.

#### 3.4 Does the compressed model still do its job?

Reconstruction error tells us how far the weights moved. It does not tell us whether the model still works and those are genuinely different questions: a network can absorb a lot of weight noise, right up to the point where it cannot. So we measure the model on its actual task.

Our probe is a vocabulary of one hundred words. From the uncompressed model we record, for every word, its five nearest neighbours in embedding space. Then we compress the model and ask two questions:

- **Agreement with the original model:** the cosine similarity between a word's embedding before and after compression. A value of 1.00 means the compressed model produced exactly the same vector.
- **Top-5 neighbours kept:** the fraction of each word's original five neighbours that are still among its five nearest neighbours. This is the metric a product manager cares about: it is the answer to "would the search results change?"

The second one is deliberately unforgiving. It ignores how the vectors moved and asks only whether the *ranking* the user sees survived.

In [ ]:
VOCAB = [
    "father", "mother", "son", "daughter", "brother", "sister", "uncle", "aunt",
    "king", "queen", "prince", "princess", "duke", "duchess", "emperor", "empress",
    "doctor", "nurse", "engineer", "teacher", "lawyer", "chef", "scientist", "artist",
    "pilot", "farmer", "soldier", "musician",
    "paris", "london", "zurich", "bern", "geneva", "berlin", "tokyo", "rome",
    "france", "japan", "germany", "switzerland", "italy", "spain", "china", "brazil",
    "car", "bicycle", "train", "airplane", "truck", "motorcycle", "helicopter", "boat",
    "router", "antenna", "battery", "processor", "sensor", "satellite", "modem", "transistor",
    "apple", "orange", "banana", "grape", "pizza", "pasta", "sushi", "bratwurst",
    "bread", "cheese", "coffee", "chocolate",
    "cat", "dog", "horse", "rabbit", "eagle", "salmon", "elephant", "wolf",
    "river", "mountain", "forest", "ocean", "desert", "island", "valley", "glacier",
    "good", "bad", "cold", "hot", "fast", "slow", "heavy", "light",
    "happy", "sad", "loud", "quiet",
]

TOP_K = 5


def embed_vocabulary():
    return model.encode(VOCAB, normalize_embeddings=True, show_progress_bar=False)


def nearest_neighbours(embeddings, k=TOP_K):
    similarity = embeddings @ embeddings.T
    np.fill_diagonal(similarity, -np.inf)      # a word is not its own neighbour
    return np.argsort(-similarity, axis=1)[:, :k]


restore_original()
REFERENCE_EMBEDDINGS = embed_vocabulary()
REFERENCE_NEIGHBOURS = nearest_neighbours(REFERENCE_EMBEDDINGS)


def evaluate_current_model():
    """Compare whatever is currently loaded against the uncompressed reference."""
    embeddings = embed_vocabulary()
    agreement = float(np.mean(np.sum(embeddings * REFERENCE_EMBEDDINGS, axis=1)))
    neighbours = nearest_neighbours(embeddings)
    kept = float(np.mean([len(set(new) & set(old)) / TOP_K
                          for new, old in zip(neighbours, REFERENCE_NEIGHBOURS)]))
    return agreement, kept


print(f"{len(VOCAB)} words embedded. Neighbours found by the uncompressed model:\n")
for word in ("king", "zurich", "antenna", "pizza", "glacier", "cold"):
    i = VOCAB.index(word)
    print(f"  {word:10s} -> {', '.join(VOCAB[j] for j in REFERENCE_NEIGHBOURS[i])}")

Those neighbour lists are the model doing something sensible: `king` finds royalty, `zurich` finds Swiss cities, `antenna` finds other hardware. Whatever survives compression has to keep producing lists like these.

Now we compress the whole model and measure. We also need an honest file size, which means counting the scales as well as the weights and leaving the parameters we did not quantize at 16 bits.

In [ ]:
def quantized_size_mb(bits, group_size=None):
    """File size when the large matrices use `bits` and everything else stays at 16 bits."""
    total_bits = 0
    for name, parameter in encoder.named_parameters():
        if is_matrix(name, parameter):
            total_bits += parameter.numel() * bits
            if group_size:
                total_bits += parameter.numel() // group_size * 16   # one FP16 scale per group
        else:
            total_bits += parameter.numel() * 16
    return total_bits / 8 / 1e6


def apply_quantization(bits, group_size=None):
    restore_original()
    with torch.no_grad():
        for name, parameter in encoder.named_parameters():
            if is_matrix(name, parameter):
                parameter.copy_(quantize_dequantize(ORIGINAL_WEIGHTS[name], bits, group_size))


restore_original()
agreement, kept = evaluate_current_model()
results = [dict(technique="none", setting="FP16 (baseline)", MB=quantized_size_mb(16),
                agreement=agreement, neighbours_kept=kept)]

for bits, group_size in [(8, None), (8, 128), (4, None), (4, 128), (4, 32), (3, 128), (2, 128)]:
    apply_quantization(bits, group_size)
    agreement, kept = evaluate_current_model()
    results.append(dict(
        technique="quantization",
        setting=f"INT{bits} " + ("per-tensor" if group_size is None else f"group={group_size}"),
        MB=quantized_size_mb(bits, group_size),
        agreement=agreement,
        neighbours_kept=kept,
    ))

restore_original()
print(pd.DataFrame(results).set_index("setting").round(3))

There is a lot in this table:

**8 bits is essentially free:** with groups, the model is half the size, agreement with the original is 1.00 and almost every neighbour ranking survives. If someone offers you an 8-bit model, there is very little to deliberate about.

**4 bits is where the decision lives:** at `INT4 group=128` the model is roughly a quarter of the FP16 baseline (an eighth of the FP32 model it was trained as) and still keeps most of the neighbour rankings. It is no longer free, but it is a trade a product can absorb, which is precisely why this is the format your Pi runs.

**The grouping decision matters more than one bit of precision:** compare `INT4 per-tensor` with `INT4 group=128`. The file sizes differ by about three percent. The quality does not differ by three percent. One of them works and the other is destroyed. The same number of bits, spent differently, is the difference between a shippable model and a broken one.

**Smaller groups cost bytes too:** `group=32` stores four times as many scales as `group=128`, so its file is about 9 % *larger*. It is the only 4-bit setting here that is bigger than another 4-bit setting. In exchange it does recover a little quality. Whether that trade is worth taking is not obvious but rather a genuine judgment. It can only be made from numbers measured on the model you intend to ship.

**Below 4 bits the floor gives way:** from 3 bits the metrics fall off and at 2 bits the model has stopped being the model. Note the shape of that decline: quality does not decay smoothly with size, it holds up and then collapses. This is characteristic and it is the reason compression choices should be measured.

#### 3.5 Quantizing after training, or training for quantization

Everything above is **post-training quantization** (PTQ): take a finished model, round its weights, ship it. It needs no training data and runs in minutes, which is why it dominates in practice.

The alternative is **quantization-aware training** (QAT). Here the rounding is simulated during training: the model performs its forward pass with rounded weights, so the optimizer can see the damage and route around it, learning weight values that survive the grid. It costs a training run, but it buys back much of the quality lost at 4 bits and below.

You have been using a QAT model all along. The `-it-qat` suffix in `gemma3:1b-it-qat` means *instruction-tuned, quantization-aware trained*: Google trained this model knowing it would be quantized. That suffix is a claim about how the model was made small and it is the reason a 4-bit Gemma behaves better than a naive 4-bit conversion of the same network would.

Further reading: [Jacob et al., 2018 "Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference"](https://arxiv.org/pdf/1712.05877) and [Dettmers et al., 2022 "LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale"](https://arxiv.org/pdf/2208.07339), which is where the outlier problem from Section 3.3 was characterized at scale.

### 4. Pruning: Fewer Weights

Quantization keeps every weight and describes each one more cheaply. **Pruning** takes the opposite route: it decides that some weights were not earning their place and deletes them.

The usual criterion is magnitude. A weight close to zero contributes little to any output it participates in, so removing it should be survivable. There are two ways to do the removing and the difference between them is the whole story:

- **Unstructured pruning** zeroes individual weights wherever they happen to be. It is gentle on quality, because it can always pick exactly the least useful weights.
- **Structured pruning** removes entire units, for instance a whole output neuron and therefore a whole row of the matrix. It is much harsher on quality, because it has to discard useful weights that share a row with useless ones.

We measure both, exactly as before. Pruning is applied only to the transformer layers; the vocabulary table is left alone, since deleting rows there would simply delete words.

In [ ]:
def prune_unstructured(W, sparsity):
    """Set the smallest weights to zero, wherever in the matrix they sit."""
    if sparsity <= 0:
        return W.clone()
    k = int(round(sparsity * W.numel()))
    threshold = W.abs().flatten().kthvalue(k).values
    return W * (W.abs() > threshold)


def prune_structured(W, sparsity):
    """Remove whole output neurons: entire rows of the matrix disappear."""
    if sparsity <= 0:
        return W.clone()
    k = int(round(sparsity * W.shape[0]))
    weakest = torch.argsort(W.norm(dim=1))[:k]
    pruned = W.clone()
    pruned[weakest] = 0.0
    return pruned


def pruned_size_mb(sparsity, structured):
    """Unstructured sparsity still needs one bit per weight to record what was removed."""
    total_bits = 0
    for name, parameter in encoder.named_parameters():
        if is_linear(name, parameter):
            total_bits += parameter.numel() * (1 - sparsity) * 16
            if not structured:
                total_bits += parameter.numel() * 1
        else:
            total_bits += parameter.numel() * 16
    return total_bits / 8 / 1e6


def apply_pruning(sparsity, structured):
    restore_original()
    prune = prune_structured if structured else prune_unstructured
    with torch.no_grad():
        for name, parameter in encoder.named_parameters():
            if is_linear(name, parameter):
                parameter.copy_(prune(ORIGINAL_WEIGHTS[name], sparsity))


for sparsity in (0.3, 0.5, 0.7, 0.9):
    apply_pruning(sparsity, structured=False)
    agreement, kept = evaluate_current_model()
    results.append(dict(technique="pruning (unstructured)", setting=f"{sparsity:.0%} of weights",
                        MB=pruned_size_mb(sparsity, False),
                        agreement=agreement, neighbours_kept=kept))

for sparsity in (0.1, 0.25, 0.5):
    apply_pruning(sparsity, structured=True)
    agreement, kept = evaluate_current_model()
    results.append(dict(technique="pruning (structured)", setting=f"{sparsity:.0%} of neurons",
                        MB=pruned_size_mb(sparsity, True),
                        agreement=agreement, neighbours_kept=kept))

restore_original()
pruning_results = pd.DataFrame(results).set_index("setting")
print(pruning_results[pruning_results["technique"].str.startswith("pruning")].round(3))

Two observations:

**Unstructured pruning degrades gracefully, structured pruning does not:** removing 30 % of individual weights leaves more of the rankings intact than removing just 10 % of whole neurons. It also produces the smaller model. A neuron that looks weak on average can still carry a few weights the rest of the network depends on and structured pruning throws those away with the rest of the row. Choosing *what* to delete matters more than choosing *how much*.

**Sparsity is not the same thing as size:** look at the megabytes. Removing half the weights did not halve the file. A matrix with holes in it is still a matrix: unless we also record *where* the holes are, we cannot skip them and that bookkeeping costs about one bit per weight. Structured pruning avoids the bookkeeping entirely. A removed row is simply absent, but we already saw what it costs in quality.

The same asymmetry governs speed, and here it is even sharper. Section 1.1 established that generation speed is set by how many bytes have to be read per token. Unstructured sparsity does not reduce that number on ordinary CPU and GPU hardware, because the dense matrix multiply reads the zeros along with everything else. A model that is 50% sparse in the unstructured sense is, on a Raspberry Pi, exactly as slow as the model it came from. Structured pruning genuinely removes rows, so it genuinely removes work.

> This is why pruning has a much smaller presence in shipped mobile models than its prominence in the research literature would suggest. It offers a quality-per-parameter trade in a world where the binding constraints are bytes-per-second and bytes-in-memory.

One caveat: we pruned and measured immediately. Production pruning pipelines fine-tune afterwards, which recovers a large part of the loss. That recovery is not free either. It needs training data, a training budget and an engineering team. When comparing techniques in the next section, keep in mind that quantization is being compared against pruning *at its cheapest* and that the realistic version of pruning costs a retraining run.

Further reading: [Frankle & Carbin, 2019 "The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks"](https://arxiv.org/pdf/1803.03635) and [Frantar & Alistarh, 2023 "SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot"](https://arxiv.org/pdf/2301.00774).

### 5. Quality against Size: Putting the Decisions on One Chart

We now have every measurement in one table, which means we can stop discussing techniques and start comparing offers. Each point below is a concrete proposal: *this many megabytes, that much retained quality*. The only points worth considering are the ones on the upper-left. Anything below and to the right of another point is dominated, meaning some other choice is both smaller and better.

In [ ]:
comparison = pd.DataFrame(results)

COLORS = {
    "none": "#757575",
    "quantization": "#1b9e77",
    "pruning (unstructured)": "#d95f02",
    "pruning (structured)": "#7570b3",
}

# The two 4-bit group settings land almost on top of each other; nudge one label apart.
LABEL_OFFSETS = {"INT4 group=32": (8, -14)}

fig, ax = plt.subplots(figsize=(11, 6))

for technique, color in COLORS.items():
    group = comparison[comparison["technique"] == technique].sort_values("MB")
    if technique != "none":
        # Connect the settings that form one continuous trade-off curve.
        curve = group[~group["setting"].str.contains("per-tensor")]
        ax.plot(curve["MB"], curve["neighbours_kept"], color=color, linewidth=1.2, alpha=0.55)
    ax.scatter(group["MB"], group["neighbours_kept"], s=95, color=color,
               label=technique, edgecolor="white", zorder=3)
    for _, row in group.iterrows():
        ax.annotate(row["setting"], (row["MB"], row["neighbours_kept"]),
                    textcoords="offset points",
                    xytext=LABEL_OFFSETS.get(row["setting"], (8, 6)), fontsize=8)

ax.annotate("better", xy=(5.0, 1.03), fontsize=10, color="#555555")
ax.annotate("", xy=(2.0, 1.07), xytext=(9.5, 0.98),
            arrowprops=dict(arrowstyle="->", color="#555555"))

ax.axhline(1.0, color="#757575", linewidth=0.8, linestyle=":")
ax.set_xlabel("stored size of the model (MB)")
ax.set_ylabel(f"top-{TOP_K} neighbours kept")
ax.set_xlim(0, comparison["MB"].max() * 1.15)
ax.set_ylim(0, 1.12)
ax.set_title("Quality against size: every point is one compression decision", loc="left")
ax.legend(loc="lower right")

plt.show()

The chart settles the comparison. Take `INT4 group=128`: it is roughly three times smaller than 50 % unstructured pruning *and* keeps more of the neighbour rankings. The same holds against every other pruning setting we measured and `INT8 group=128` beats all of them too while staying essentially lossless.

Not every green point wins: `INT4 per-tensor` and `INT2` sit below the pruning curve, but those are precisely the settings Section 3 showed to be broken. Among the settings anyone would actually deploy, there is no budget at which the pruning curve is the right thing to buy on this model. Add the facts that the pruned model would additionally need a retraining run and would not generate any faster, and the case closes.

However, we need to note that this is one small encoder model and one task, so the numbers are not a universal law. Pruning here is unfinetuned. What generalizes is not the exact positions of the dots but the shape of the picture: **on memory-bound devices, spending fewer bits per weight beats keeping fewer weights.** That is why the mobile ecosystem converged on quantized models and why `ollama show` reports a quantization level rather than a sparsity level.

It is also worth noticing that the two techniques are not rivals in principle. The classic result in this area combined pruning, quantization and entropy coding in one pipeline and the industrial pipelines of today combine pruning with distillation before quantizing the result. Quantization is the last step almost every time.

Further reading: [Han et al., 2016 "Deep Compression: Compressing Deep Neural Networks with Pruning, Trained Quantization and Huffman Coding"](https://arxiv.org/pdf/1510.00149).

### 6. Distillation: Training a Small Model to Imitate a Large One

Quantization and pruning both start from a finished model and remove something from it. **Knowledge distillation** takes a different route: it trains a new, smaller model (the *student*) from scratch to reproduce the behavior of a large one (the *teacher*).

The insight is about what the student is trained on. Ordinary training uses **hard labels**: the correct next word, nothing else. Distillation instead uses the teacher's full **output distribution** as the target, its probability for every possible next token. That distribution carries much more information than the single correct answer and the extra information is sometimes called *dark knowledge*: it tells the student not just what is right, but what was nearly right and what was never plausible at all.

To make this signal usable, the teacher's distribution is softened with a temperature, the same parameter you tuned in [`05.11_Understanding_Inference_Control.ipynb`](./05.11_Understanding_Inference_Control.ipynb). At `T = 1` a confident teacher puts almost all its mass on one token and the runners-up are numerically invisible. Raising the temperature lifts them into a range where they can actually influence the student's gradients.

In [ ]:
CANDIDATES = ["battery", "signal", "price", "resolution", "temperature", "elephant"]
teacher_logits = np.array([6.0, 4.6, 3.4, 3.0, 1.4, -3.0])   # scores from a large teacher model


def softmax(scores, temperature=1.0):
    shifted = scores / temperature
    shifted = shifted - shifted.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


hard_label = np.zeros(len(CANDIDATES))
hard_label[CANDIDATES.index("battery")] = 1.0
teacher_t1 = softmax(teacher_logits, temperature=1.0)
teacher_t4 = softmax(teacher_logits, temperature=4.0)

targets = pd.DataFrame(
    {"hard label": hard_label, "teacher, T=1": teacher_t1, "teacher, T=4": teacher_t4},
    index=CANDIDATES,
)
print('Prompt: "The mobile phone has a low ..."\n')
print(targets.round(4))

fig, ax = plt.subplots(figsize=(10, 4))
offsets = np.arange(len(CANDIDATES))
for shift, column, color in [(-0.27, "hard label", "#757575"),
                             (0.00, "teacher, T=1", "#4c78a8"),
                             (0.27, "teacher, T=4", "#1b9e77")]:
    ax.bar(offsets + shift, targets[column], width=0.27, label=column, color=color)

ax.set_xticks(offsets, CANDIDATES)
ax.set_ylabel("probability")
ax.set_title('What the student is asked to reproduce: "The mobile phone has a low ..."', loc="left")
ax.legend()

plt.show()

The hard label is a single grey bar: *the answer is `battery`, everything else is equally wrong*. That last part is false and the teacher knows it. `signal` and `price` are perfectly reasonable continuations, `resolution` is plausible, and `elephant` is not a candidate at all. The hard label throws that entire ranking away.

At `T = 1` the teacher does encode the ranking, but the tail is numerically negligible. At `T = 4` the same ranking is spread across a range where every candidate contributes to the student's learning signal. The student therefore learns a structure over the vocabulary, not just a sequence of correct answers and it can learn that structure from far fewer examples than the teacher needed.

This is why distillation produces genuinely capable small models rather than merely small ones. You already have one on your Pi: `deepseek-r1:1.5b` is not a shrunken DeepSeek-R1, it is a smaller model trained on reasoning traces produced by the large one. Its ability to lay out a chain of thought, which you observed in [`05.12_Understanding_Reasoning_Models.ipynb`](./05.12_Understanding_Reasoning_Models.ipynb), was inherited rather than learned from raw text.

The catch is cost and it falls in a different place from the other techniques. Quantization takes minutes and needs no data. Distillation needs a teacher, a corpus, and a full training run. It is a decision made by whoever builds the model, not by whoever deploys it, which is exactly why it appears in your workflow as a *choice of model* rather than as a setting you can tune.

Further reading: [Hinton et al., 2015 "Distilling the Knowledge in a Neural Network"](https://arxiv.org/pdf/1503.02531) and [Sanh et al., 2019 "DistilBERT, a distilled version of BERT"](https://arxiv.org/pdf/1910.01108).

### 7. Selective Parameter Activation: Using Fewer Weights at a Time

The last technique does not make the model smaller at all. It makes the model *cheaper per token*, which is a different thing and confusing the two leads to expensive mistakes.

In a **mixture-of-experts** architecture, each layer holds many parallel sub-networks (*experts*) and a small router picks only a few of them for each token. A model might hold 30 billion parameters while activating only 3 billion for any given token. Since generation speed is set by the bytes that must be read per token, activating a tenth of the weights makes generation dramatically faster.

The catch is in Section 1.1's first constraint. The router's choice depends on the token and it may select any expert at any moment, so **every expert has to be resident in memory**. A mixture-of-experts model needs the memory of its full parameter count while delivering the speed of its active parameter count.

That trade is excellent in a data center, where memory is abundant and latency is the scarce resource. It is close to useless on a Raspberry Pi, where memory is precisely the thing you have run out of. This is the clearest case in the notebook of a technique whose value depends entirely on which constraint binds in your deployment.

`gemma3n:e2b` from [`05.07_Introduction_to_Local_LLMs.ipynb`](./05.07_Introduction_to_Local_LLMs.ipynb) is the mobile-oriented answer to this tension. The `e2b` stands for *effective 2B*: the model holds substantially more parameters than that, but is built so that only a 2B-sized subset needs to be resident for ordinary use. If you were surprised by its parameter count when you ran `ollama show` on it, this is the reason.

Further reading: [Shazeer et al., 2017 "Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer"](https://arxiv.org/pdf/1701.06538).

### 8. Checking the Story on the Raspberry Pi

Everything so far was measured on a small encoder on your laptop. Now we test whether the claims hold where it counts. Three claims are worth checking, and they are checkable in about twenty minutes:

1. The models on the Pi really are quantized, and the runtime will tell us to what.
2. A model that is half the size generates roughly twice as fast, because generation is memory-bound.
3. Buying quality with size has a steeply diminishing return, and somewhere on that curve is a defensible product decision.

#### 8.1 What the runtime says about compression

<div class="alert alert-block alert-info">

Connect to the Pi over SSH and inspect the models. Look for the `quantization` field, the parameter count, and the file size. Note down the quantization format of each model.
</div>

```sh
ollama list

ollama show gemma3:270m
ollama show gemma3:1b-it-qat
ollama show gemma3:4b-it-qat
ollama show deepseek-r1:1.5b
```

Compare the file size that `ollama list` reports against the `FP32` and `FP16` columns of the table in Section 2. The gap between them is the compression, and it is the reason the model is on the device at all.

#### 8.2 Does halving the size double the speed?

<div class="alert alert-block alert-info">

Run the same prompt on three models of very different sizes, with `--verbose` so that Ollama reports the timings. Stop the previous model before starting the next one, so that only one occupies memory at a time. Record the **eval rate** in tokens per second, and use `ollama ps` in a second SSH session to record how much memory each model actually occupies.
</div>

```sh
ollama-stop
ollama run gemma3:270m --verbose "Explain in three sentences why a smaller model runs faster on a Raspberry Pi."

ollama-stop
ollama run gemma3:1b-it-qat --verbose "Explain in three sentences why a smaller model runs faster on a Raspberry Pi."

ollama-stop
ollama run gemma3:4b-it-qat --verbose "Explain in three sentences why a smaller model runs faster on a Raspberry Pi."
```

#### 8.3 Does the extra size buy anything?

Speed is only half of the trade. To see the other half we need a quality score, so we use a small fixed test and count correct answers. Five questions is not a benchmark, but it is enough to see the shape of the curve, and it forces you to judge the outputs yourself rather than trusting a number.

<div class="alert alert-block alert-info">

Put the same five questions to each of the three models and score each answer as correct or not. Keep the decoding parameters fixed across models — use `ollama-gen` with a fixed `--seed` and a fixed `--temp`, as in [`05.11_Understanding_Inference_Control.ipynb`](./05.11_Understanding_Inference_Control.ipynb) — so that the only thing changing between runs is the model.

Suggested questions: one arithmetic word problem, one factual question, one instruction to follow a strict output format, one summary of a short paragraph you provide, and one question whose honest answer is "I do not know".
</div>

Now enter your own measurements. Replace the example values below with the numbers you recorded on the Pi.

In [ ]:
# REPLACE these example values with your own measurements from the Raspberry Pi.
EXAMPLE_VALUES = True          # set to False once you have entered your own numbers

measurements = {
    "gemma3:270m":      dict(file_mb=292,  resident_mb=800,  tokens_per_s=27.0, correct=1),
    "gemma3:1b-it-qat": dict(file_mb=1000, resident_mb=1900, tokens_per_s=9.5,  correct=3),
    "gemma3:4b-it-qat": dict(file_mb=3300, resident_mb=4200, tokens_per_s=2.6,  correct=4),
}

pi = pd.DataFrame(measurements).T

# If generation really is memory-bound, then every token costs one full read of the
# weights, so file size x tokens per second should land near the memory bandwidth --
# and should be roughly the SAME number for all three models.
pi["effective MB/s"] = (pi["file_mb"] * pi["tokens_per_s"]).round(0)

if EXAMPLE_VALUES:
    print("NOTE: these are example values, not measurements. Replace them and set "
          "EXAMPLE_VALUES = False.\n")
print(pi)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), constrained_layout=True)

for ax, column, ylabel, color in [
    (axes[0], "tokens_per_s", "generation speed (tokens/s)", "#4c78a8"),
    (axes[1], "correct", "correct answers (out of 5)", "#1b9e77"),
]:
    ax.plot(pi["file_mb"], pi[column], color=color, alpha=0.4, zorder=2)
    ax.scatter(pi["file_mb"], pi[column], s=95, color=color, zorder=3)
    for name, row in pi.iterrows():
        ax.annotate(name, (row["file_mb"], row[column]),
                    textcoords="offset points", xytext=(0, 10), fontsize=8, ha="center")
    ax.set_xlabel("model file size (MB)")
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, pi["file_mb"].max() * 1.3)

axes[0].set_ylim(0, pi["tokens_per_s"].max() * 1.35)
axes[1].set_ylim(0, 6)
axes[0].set_title("Speed against size", loc="left")
axes[1].set_title("Quality against size", loc="left")

plt.show()

The `effective MB/s` column is the interesting one, because it is a test rather than a description. Section 1.1 claimed that generating one token requires reading every weight from memory once. If that is true, then multiplying a model's size by its generation speed should recover roughly the same number: the memory bandwidth of the Pi. Three models that differ by a factor of ten in size should agree on that one figure.

Check whether yours do. If they land in the roughly same range, you have measured the memory wall directly, and you have your own evidence for why a 4-bit model is faster than a 16-bit one. If the smallest model falls well short, that is informative too: at very small sizes the fixed overheads of sampling and of the runtime start to dominate, and the model is no longer purely memory-bound.

The two panels then show the trade-off itself. Speed falls roughly in proportion to size, while quality bends: the step from 270M to 1B buys a great deal, and the step from 1B to 4B buys less while costing three times the memory and several times the latency. Where those curves cross is a product decision rather than a technical one. A model answering safety-relevant questions should be bought further to the right than one suggesting autocomplete text.

#### 8.4 Matching the technique to the model

Every technique in this notebook is visible in the models you already have. Naming which is which is a useful check that the concepts have landed.

| Model | Technique that made it small | How you can tell |
|:------|:-----------------------------|:-----------------|
| `gemma3:1b-it-qat` | quantization, trained for it (QAT) | the `qat` suffix; `ollama show` reports a 4-bit format |
| `deepseek-r1:1.5b` | distillation from a much larger reasoning model | it reasons far better than its size suggests |
| `gemma3n:e2b` | selective parameter activation | `ollama show` reports more parameters than the `2b` in its name |
| `gemma3:270m` | none of the above, it was simply trained small | quality drops off in a way compression alone does not explain |

<div class="alert alert-block alert-info">

Run `ollama show` on all four and find the evidence for each row yourself. One row is deliberately harder to confirm than the others.
</div>

Pruning is absent from that table, and its absence is informative. Pruned models are rarely released as products in their own right; pruning appears inside industrial pipelines as a step before distillation and quantization, not as the label on the box. That is the same conclusion Section 5 reached from measurements, arriving here from the direction of what the market actually ships.

Further reading: [Muralidharan et al., 2024 "Compact Language Models via Pruning and Knowledge Distillation"](https://arxiv.org/pdf/2407.14679), which describes exactly such a pipeline.

### 9. Summarizing Remarks

Making a model smaller is not one decision but a small set of them, and they have different costs, different beneficiaries, and different failure modes.

**Quantization is the default, and deservedly so.** It is cheap to apply, needs no training data, shrinks the file and speeds up generation in the same move. Eight bits is close to free. Four bits is the working compromise the mobile ecosystem has settled on. Below four bits, quality does not decline gracefully; it collapses.

**How the bits are spent matters as much as how many there are.** The single sharpest result in this notebook is that two 4-bit models of almost identical size can differ between working and worthless, depending only on whether the scale is shared across a whole matrix or across small groups. Whenever a compression scheme has to accommodate rare extreme values, the typical values pay for it.

**Pruning trades in the wrong currency for mobile deployment.** It reduces parameter counts, but mobile devices are constrained by bytes and by bandwidth. Unstructured sparsity saves neither without special hardware support; structured sparsity saves both but costs much more quality. It remains valuable inside a pipeline that can afford to retrain afterwards.

**Distillation and selective activation are decisions made upstream of you.** You cannot apply them to a model you downloaded; you consume them by choosing which model to download. Recognising which technique produced a candidate model tells you a great deal about how it will behave under further compression and where its quality will run out.

**The general management point.** Every technique here converts one resource into another: precision into memory, parameters into quality, training compute into inference cost. None of them creates capability. The competent decision is therefore not "how small can this get" but "which resource is actually scarce in my deployment, and what is the cheapest thing I can spend to relieve it". A team that answers that question with measurements from its own device will consistently beat one that copies a configuration from a blog post.

And there is a reason this notebook sits at the center of the mobile AI part of the course. Every other capability you will build on the Pi — retrieval, agents, tool use — is built on top of a model that had to be made to fit first. The compression decision is upstream of all of them.

### Further Reading and Literature

[01] Hinton, G., Vinyals, O., & Dean, J. (2015). **Distilling the Knowledge in a Neural Network.** NeurIPS Deep Learning Workshop. [Paper](https://arxiv.org/abs/1503.02531)

[02] Han, S., Mao, H., & Dally, W. J. (2016). **Deep Compression: Compressing Deep Neural Networks with Pruning, Trained Quantization and Huffman Coding.** ICLR. [Paper](https://arxiv.org/abs/1510.00149)

[03] Shazeer, N., Mirhoseini, A., Maziarz, K., Davis, A., Le, Q., Hinton, G., & Dean, J. (2017). **Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer.** ICLR. [Paper](https://arxiv.org/abs/1701.06538)

[04] Jacob, B., Kligys, S., Chen, B., Zhu, M., Tang, M., Howard, A., Adam, H., & Kalenichenko, D. (2018). **Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference.** CVPR. [Paper](https://arxiv.org/abs/1712.05877)

[05] Frankle, J., & Carbin, M. (2019). **The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks.** ICLR. [Paper](https://arxiv.org/abs/1803.03635)

[06] Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). **DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter.** NeurIPS EMC^2 Workshop. [Paper](https://arxiv.org/abs/1910.01108)

[07] Wang, W., Wei, F., Dong, L., Bao, H., Yang, N., & Zhou, M. (2020). **MiniLM: Deep Self-Attention Distillation for Task-Agnostic Compression of Pre-Trained Transformers.** NeurIPS. [Paper](https://arxiv.org/abs/2002.10957) — the model used throughout this notebook is itself a distilled one.

[08] Dettmers, T., Lewis, M., Belkada, Y., & Zettlemoyer, L. (2022). **LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale.** NeurIPS. [Paper](https://arxiv.org/abs/2208.07339)

[09] Frantar, E., Ashkboos, S., Hoefler, T., & Alistarh, D. (2023). **GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers.** ICLR. [Paper](https://arxiv.org/abs/2210.17323)

[10] Frantar, E., & Alistarh, D. (2023). **SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot.** ICML. [Paper](https://arxiv.org/abs/2301.00774)

[11] Lin, J., Tang, J., Tang, H., Yang, S., Dang, X., & Han, S. (2024). **AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration.** MLSys. [Paper](https://arxiv.org/abs/2306.00978)

[12] Muralidharan, S., Sreenivas, S. T., Joshi, R., Chochowski, M., Patwary, M., Shoeybi, M., Catanzaro, B., Kautz, J., & Molchanov, P. (2024). **Compact Language Models via Pruning and Knowledge Distillation.** NeurIPS. [Paper](https://arxiv.org/abs/2407.14679)

---
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. Many questions ask for concrete values or outputs from your own notebook run; generic answers without run-specific evidence may receive reduced credit. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Section 2 asks you to read a table. Which is the smallest model in the list that does *not* fit into the Pi's usable memory at `FP16`, and which precision is the first one that makes it fit?** Answer:

(add your answer here)

**Generation speed on the Pi is limited by memory bandwidth rather than by the processor. Explain in your own words why this makes quantization a speed improvement and not only a storage improvement.** Answer:

(add your answer here)

**Report the numbers from your own run: what is the file size and the top-5 neighbour score for `INT4 per-tensor` and for `INT4 group=128`? Why does a difference of a few percent in size produce such a different outcome?** Answer:

(add your answer here)

**`INT4 group=32` stores more scales than `INT4 group=128`. Look at the two file sizes in your table. Was the extra fidelity worth the extra bytes? Justify your answer with both columns.** Answer:

(add your answer here)

**In your run, at how many bits does quality first fall below roughly 60 % of neighbours kept? Is the decline gradual or sudden, and what does your answer imply about choosing a compression level by interpolation rather than by measurement?** Answer:

(add your answer here)

**Compare 50 % unstructured pruning with `INT4 group=128` using your own numbers for size and quality. Which would you deploy, and what is the strongest argument someone could make for the other one?** Answer:

(add your answer here)

**Removing 50 % of the weights did not remove 50 % of the megabytes. Explain where the missing savings went, and why structured pruning does not have this problem.** Answer:

(add your answer here)

**We pruned without any retraining afterwards. Explain how this makes the comparison in Section 5 unfair to pruning, and describe what a fair comparison would have cost to run.** Answer:

(add your answer here)

**What information does a teacher's softened output distribution contain that a hard label does not? Use one concrete pair of tokens from the Section 6 output to make the point.** Answer:

(add your answer here)

**A mixture-of-experts model activates 3B of its 30B parameters per token. Explain why this is attractive in a data center and unattractive on the Raspberry Pi.** Answer:

(add your answer here)

**Report your Pi measurements: the file size, resident memory and tokens per second for each of the three models. Did halving the file size roughly double the generation speed? If not, suggest what else was consuming the time.** Answer:

(add your answer here)

**Look at your `effective MB/s` column. How close are the three values to each other, and what does that tell you about whether generation on the Pi is limited by memory or by the processor?** Answer:

(add your answer here)

**Report your five-question quality scores. At which step up in model size did you get the least additional quality per megabyte, and would you have made a different choice for a safety-critical application?** Answer:

(add your answer here)

**Using `ollama show`, give the evidence you found for each row of the table in Section 8.4. Which row was hardest to confirm, and why?** Answer:

(add your answer here)

**Optional: You are advising a company that wants an offline assistant on a handheld device with 4 GB of RAM. Using the evidence from this notebook, state which compression technique you would require of a vendor, which claim you would ask them to demonstrate, and what measurement you would run to verify it.** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)